# Can we predict 12-month loan default?

**Question:** from information known at application time, can we predict `defaulted` (1 = defaulted within 12 months of funding)?

**Plan:** check data quality → explore → hold out a test set → compare a baseline, logistic regression and gradient boosting using cross-validation → evaluate once on the test set → judge whether the predictions are useful in practice.

Metrics: the target is imbalanced (~10% defaults), so accuracy is misleading. I use ROC AUC (ranking quality), PR AUC / lift (usefulness for flagging risky loans) and calibration.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
SEED = 42
pd.set_option("display.width", 120)

## 1. Data and quality checks

In [ ]:
df = pd.read_csv("data.csv")
dd = pd.read_csv("columns.csv")
display(dd)
print(df.shape)
display(df.head())
print("missing values:\n", df.isna().sum().to_string())
print("duplicate rows:", df.duplicated().sum(), "| duplicate ids:", df.application_id.duplicated().sum())
display(df.describe().T)

No missing values or duplicates. `application_id` is an identifier, so it is not used as a feature (checked below that it carries no signal). There is no date column, so a time-based split is impossible; I use a stratified random split, which is optimistic if the real population drifts over time.

In [ ]:
print(f"Default rate: {df.defaulted.mean():.3%}  ({df.defaulted.sum()} of {len(df)})")
print("Share with employment_years == 0:", (df.employment_years == 0).mean().round(3))
print("Rows with 0 credit lines:", (df.credit_lines == 0).sum())
# does the id look informative? (default rate by id decile)
print(df.groupby(pd.qcut(df.application_id, 10), observed=True).defaulted.mean().round(3).to_string())

## 2. Exploration

In [ ]:
num = ["income_k","debt_ratio","credit_lines","age","employment_years","loan_amount_k"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), num):
    q = pd.qcut(df[c], 10, duplicates="drop")
    g = df.groupby(q, observed=True).defaulted.mean()
    ax.plot(range(len(g)), g.values, marker="o")
    ax.axhline(df.defaulted.mean(), ls="--", c="grey")
    ax.set_title(f"Default rate by {c} decile"); ax.set_xlabel("decile (low → high)"); ax.set_ylim(0, None)
plt.tight_layout(); plt.show()
print(df.groupby("region").defaulted.agg(["mean","size"]).round(3))
print(df[num+["defaulted"]].corr(method="spearman")["defaulted"].round(3))

`debt_ratio` is the strongest signal; lower income, shorter employment and larger loans also raise risk. `age`, `credit_lines`, `region` and `application_id` look uninformative on their own (region differences are within sampling noise at n≈2000 per region). Since income, loan size and debt ratio are plausibly related, I add `loan_to_income` and `log_income` as candidate features.

## 3. Split and features

In [ ]:
def add_features(d):
    d = d.drop(columns=["application_id"]).copy()
    d["loan_to_income"] = d.loan_amount_k / d.income_k
    d["log_income"] = np.log(d.income_k)
    return d

X, y = add_features(df.drop(columns="defaulted")), df.defaulted
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(X_train.shape, X_test.shape, y_train.mean().round(4), y_test.mean().round(4))
num_cols = [c for c in X.columns if c != "region"]

## 4. Model comparison (5-fold stratified CV on the training set only)

In [ ]:
prep_lin = ColumnTransformer([("num", StandardScaler(), num_cols), ("reg", OneHotEncoder(drop="first"), ["region"])])
prep_tree = ColumnTransformer([("num", "passthrough", num_cols), ("reg", OneHotEncoder(drop="first"), ["region"])])
models = {
    "Prior (no features)": DummyClassifier(strategy="prior"),
    "Logistic, debt_ratio only": make_pipeline(ColumnTransformer([("d","passthrough",["debt_ratio"])]), LogisticRegression()),
    "Logistic regression": make_pipeline(prep_lin, LogisticRegression(C=1.0, max_iter=1000)),
    "Gradient boosting": make_pipeline(prep_tree, HistGradientBoostingClassifier(
        learning_rate=0.03, max_iter=300, max_depth=3, min_samples_leaf=40, l2_regularization=1.0, random_state=SEED)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=["roc_auc","average_precision","neg_brier_score"])
    rows.append({"model": name,
                 "ROC AUC": r["test_roc_auc"].mean(), "± AUC (fold std)": r["test_roc_auc"].std(),
                 "PR AUC": r["test_average_precision"].mean(), "Brier": -r["test_neg_brier_score"].mean()})
cv_res = pd.DataFrame(rows).set_index("model").round(4)
cv_res

## 5. Final evaluation on the held-out test set

In [ ]:
final = {k: models[k].fit(X_train, y_train) for k in ["Logistic regression", "Gradient boosting"]}
base = y_test.mean()
rows = []
for k, m in final.items():
    p = m.predict_proba(X_test)[:, 1]
    rng = np.random.default_rng(SEED); aucs = []
    for _ in range(500):
        i = rng.integers(0, len(y_test), len(y_test))
        aucs.append(roc_auc_score(y_test.iloc[i], p[i]))
    rows.append({"model": k, "ROC AUC": roc_auc_score(y_test, p), "AUC 95% CI": f"{np.percentile(aucs,2.5):.3f}–{np.percentile(aucs,97.5):.3f}",
                 "PR AUC": average_precision_score(y_test, p), "Brier": brier_score_loss(y_test, p),
                 "Brier (prior)": brier_score_loss(y_test, np.full(len(y_test), y_train.mean()))})
display(pd.DataFrame(rows).set_index("model").round(4))
print(f"Test base rate (= PR AUC of a random ranking): {base:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for k, m in final.items():
    p = m.predict_proba(X_test)[:, 1]
    f, t, _ = roc_curve(y_test, p); ax[0].plot(f, t, label=k)
    pr, rc, _ = precision_recall_curve(y_test, p); ax[1].plot(rc, pr, label=k)
    fp, mp = calibration_curve(y_test, p, n_bins=8, strategy="quantile"); ax[2].plot(mp, fp, marker="o", label=k)
ax[0].plot([0,1],[0,1],"k--"); ax[0].set(title="ROC", xlabel="FPR", ylabel="TPR")
ax[1].axhline(base, c="k", ls="--"); ax[1].set(title="Precision–recall", xlabel="recall", ylabel="precision")
ax[2].plot([0,.5],[0,.5],"k--"); ax[2].set(title="Calibration", xlabel="mean predicted", ylabel="observed rate")
for a in ax: a.legend()
plt.tight_layout(); plt.show()

## 6. Is it useful? Lift when ranking applicants by predicted risk

In [ ]:
best = final["Logistic regression"]
p = best.predict_proba(X_test)[:, 1]
t = pd.DataFrame({"p": p, "y": y_test.values})
t["decile"] = pd.qcut(t.p.rank(method="first"), 10, labels=range(1, 11))
lift = t.groupby("decile", observed=True).agg(n=("y","size"), predicted=("p","mean"), observed_default_rate=("y","mean")).round(3)
display(lift)
top = t.sort_values("p", ascending=False)
for frac in (0.1, 0.2, 0.3):
    k = int(len(top) * frac); s = top.head(k)
    print(f"Flag top {frac:.0%} riskiest: precision {s.y.mean():.1%} (base {base:.1%}, lift {s.y.mean()/base:.1f}x), captures {s.y.sum()/t.y.sum():.0%} of defaults")

## 7. What drives the prediction?

In [ ]:
coef = pd.Series(best[-1].coef_[0], index=best[:-1].get_feature_names_out()).sort_values()
print("Logistic coefficients (numerics standardised; positive = higher default risk)")
display(coef.round(3).to_frame("coef"))
pi = pd.Series(permutation_importance(final["Gradient boosting"], X_test, y_test, scoring="roc_auc",
                                      n_repeats=20, random_state=SEED).importances_mean, index=X_test.columns).sort_values()
pi.plot.barh(title="Permutation importance (drop in test AUC), gradient boosting")
plt.tight_layout(); plt.show()

## 8. Conclusion

*(Numbers below are from the run with `SEED = 42`; re-running reproduces them.)*

**Yes, but only moderately.** Default is predictable well enough to rank applicants by risk, not well enough to call individual outcomes.

- **Discrimination:** held-out ROC AUC ≈ 0.78 (95% bootstrap CI ≈ 0.74–0.81) for logistic regression, versus 0.50 for no information. PR AUC ≈ 0.36 versus a 0.10 base rate.
- **Practical value:** flagging the riskiest 10% of applicants yields a ~37% default rate (3.5× base) and catches ~35% of all defaults; the riskiest 30% catches ~65%. The bottom 20% defaults at ~1–2%.
- **Calibration:** predicted probabilities track observed rates reasonably well (decile table), so the scores can be read as risk estimates, though the 9th decile is somewhat over-predicted and each decile has only 200 loans.
- **Model choice:** simple logistic regression beat gradient boosting in CV (0.779 vs 0.761) and on the test set (0.775 vs 0.763). The signal looks mostly additive, and with ~840 defaults there is little data for flexible models. Prefer the logistic model: it is simpler and easy to explain.
- **Drivers:** higher `debt_ratio` and larger loan relative to income raise risk; higher income and longer employment lower it. `age`, `credit_lines` and `application_id` carry essentially no signal; the region effect is small (north slightly lower in the coefficients) and not clearly distinguishable from noise.

**Caveats**
- At a ~10% base rate, even the top-risk group is mostly non-defaulters (precision ≈ 37%), so this is a triage/pricing tool, not an auto-decline rule. Any cut-off should be chosen from the business cost of a default versus a rejected good loan, which I don't have.
- No dates are available, so the split is random, not out-of-time. Performance on future cohorts may be lower if the population or economy shifts.
- Only funded loans are observed (`defaulted` is measured "after funding"), so this has not been validated for applicants who were rejected (selection bias).
- Using `region`/`age` in actual lending decisions may raise fairness or regulatory issues; `age` showed no predictive value here, so it can be dropped.
- Test set is 2,000 rows (~209 defaults), so the differences between the two models (~0.01 AUC) are within noise; the ranking logistic > boosting is suggestive, not proven.

**Next steps if more accuracy is needed:** richer features (payment history, bureau data, loan term/purpose), more data, and a time-based validation.